# TC5061 · Entrenamiento parametrizado y seguimiento con MLflow

**Alumna:** Martha Alicia Villalobos Facundo · **Matrícula:** A01840063  
**Materia:** Operaciones de aprendizaje automático




In [1]:
from pathlib import Path
import os
import subprocess
import sys

WORK = Path('/content/tc5061_mlflow')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('Carpeta de trabajo:', WORK)
print('Python de Colab:', sys.version)


Carpeta de trabajo: /content/tc5061_mlflow
Python de Colab: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## 3. Archivos de trabajo



### Archivo: requirements.txt

In [2]:
%%writefile requirements.txt
mlflow==2.22.0
scikit-learn==1.6.1
numpy==2.1.3
pandas==2.2.3
scipy==1.15.2
matplotlib==3.10.1
requests==2.32.3
SQLAlchemy==2.0.40


Writing requirements.txt


### Archivo: train.py

In [3]:
%%writefile train.py
"""Entrenamiento reproducible del baseline Online Shoppers. Ejecutar: python train.py."""
import os

# Limitar hilos antes de importar las librerías numéricas.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[variable] = '1'
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('.matplotlib'))

import argparse
import hashlib
import io
import json
import platform
import random
import subprocess
import sys
import warnings
import zipfile
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import requests
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

EXPERIMENT = 'TC5061_Online_Shoppers_Regresion_Logistica_Comparacion'
DATA_URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'
FEATURES = ['Administrative', 'Administrative_Duration', 'Informational',
            'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
            'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
            'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__,
                                    formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument('--C', type=float, default=1.0, help='Inverso de la regularización; debe ser positivo')
    parser.add_argument('--max-iter', type=int, default=1000, help='Máximo de iteraciones del optimizador')
    parser.add_argument('--class-weight', choices=['balanced', 'none'], default='balanced', help='Ponderación de clases')
    parser.add_argument('--tol', type=float, default=1e-4, help='Tolerancia de convergencia positiva')
    parser.add_argument('--seed', type=int, default=42, help='Semilla para Python, NumPy y scikit-learn')
    parser.add_argument('--data-path', default='data/online_shoppers_intention.csv', help='CSV de UCI; se descarga si falta')
    parser.add_argument('--tracking-uri', default='sqlite:///mlflow.db', help='Almacén de MLflow')
    parser.add_argument('--experiment', default=EXPERIMENT, help='Nombre del experimento')
    parser.add_argument('--run-name', default='baseline', help='Nombre de la corrida')
    parser.add_argument('--batch-id', default='manual', help='Etiqueta para separar grupos de comparación')
    parser.add_argument('--role', default='comparacion', choices=['comparacion', 'replica'], help='Finalidad de la corrida')
    args = parser.parse_args()
    if args.C <= 0 or args.tol <= 0 or args.max_iter < 1 or not 0 <= args.seed < 2**32:
        parser.error('C y tol deben ser positivos, max-iter >= 1 y 0 <= seed < 2**32.')
    return args


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_data(path):
    """Descargar el CSV original, validar columnas y aplicar la limpieza del baseline."""
    path = Path(path)
    if not path.exists():
        response = requests.get(DATA_URL, timeout=90)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            matches = [n for n in archive.namelist() if Path(n).name == 'online_shoppers_intention.csv']
            if len(matches) != 1:
                raise ValueError('No se encontró un único CSV de Online Shoppers en UCI.')
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(matches[0]))
    raw = pd.read_csv(path)
    if set(raw.columns) != set(FEATURES + ['Revenue']):
        raise ValueError('El CSV no contiene las 17 características y Revenue del baseline.')
    clean = raw.drop_duplicates().copy()
    for column in ['Weekend', 'Revenue']:
        # Evitar convertir silenciosamente texto desconocido en True.
        converted = clean[column].astype(str).str.strip().str.lower().map(
            {'true': 1, 'false': 0, '1': 1, '0': 0})
        if converted.isna().any():
            raise ValueError(f'Valores booleanos no reconocidos en {column}.')
        clean[column] = converted.astype(int)
    metadata = {'raw_rows': len(raw), 'clean_rows': len(clean),
                'duplicates_removed': len(raw) - len(clean), 'data_sha256': sha256(path)}
    return clean[FEATURES], clean['Revenue'], metadata


def split_data(X, y, seed):
    """Mantener el holdout estratificado 80/20 del notebook original."""
    return train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)


def build_pipeline(X_train, args):
    """Ajustar imputación, escalado y codificación exclusivamente en entrenamiento."""
    numeric = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical = X_train.select_dtypes(exclude=np.number).columns.tolist()
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                         ('scaler', RobustScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore'))])
    preprocess = ColumnTransformer([('num', num_pipe, numeric), ('cat', cat_pipe, categorical)])
    model = LogisticRegression(C=args.C, max_iter=args.max_iter,
                               class_weight=None if args.class_weight == 'none' else 'balanced',
                               tol=args.tol, random_state=args.seed, solver='lbfgs', penalty='l2')
    return Pipeline([('preprocess', preprocess), ('model', model)])


def fit_model(pipeline, X_train, y_train):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        pipeline.fit(X_train, y_train)
    messages = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
    return pipeline, messages


def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    metrics = {'accuracy': accuracy_score(y_test, pred),
               'precision': precision_score(y_test, pred, zero_division=0),
               'recall': recall_score(y_test, pred, zero_division=0),
               'f1': f1_score(y_test, pred, zero_division=0),
               'roc_auc': roc_auc_score(y_test, prob)}
    return metrics, pred, prob


def save_artifacts(folder, metrics, model, X_train, X_test, y_test, pred, prob, messages):
    (folder / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    report = classification_report(y_test, pred, digits=5, zero_division=0)
    (folder / 'classification_report.txt').write_text(report, encoding='utf-8')
    pd.DataFrame({'row_index': X_test.index, 'real': y_test.to_numpy(),
                  'prediction': pred, 'probability': prob}).to_csv(folder / 'predictions.csv', index=False)
    (folder / 'split.json').write_text(json.dumps({'train': X_train.index.tolist(),
                                                 'test': X_test.index.tolist()}), encoding='utf-8')
    (folder / 'convergence.json').write_text(json.dumps({'warnings': messages,
         'n_iter': model.named_steps['model'].n_iter_.tolist()}, indent=2), encoding='utf-8')
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['No compra', 'Compra'], cmap='Blues')
    plt.title('Matriz de confusión — Online Shoppers')
    plt.tight_layout()
    plt.savefig(folder / 'confusion_matrix.png', dpi=160)
    plt.close()
    freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    (folder / 'environment_freeze.txt').write_text(freeze, encoding='utf-8')
    (folder / 'python_version.txt').write_text(platform.python_version(), encoding='utf-8')


def main():
    args = parse_args()
    random.seed(args.seed)
    np.random.seed(args.seed)
    mlflow.set_tracking_uri(args.tracking_uri)
    mlflow.set_experiment(args.experiment)
    with mlflow.start_run(run_name=args.run_name) as run:
        mlflow.set_tags({'batch_id': args.batch_id, 'role': args.role,
                         'baseline_source': 'TC5061_Notebook_Baseline_VillalobosMartha.ipynb'})
        for name, value in vars(args).items():
            mlflow.log_param(name, value)
        for name, value in {'model': 'LogisticRegression', 'solver': 'lbfgs', 'penalty': 'l2',
                            'test_size': 0.20, 'scaler': 'RobustScaler', 'encoding': 'OneHotEncoder',
                            'train_sha256': sha256(__file__)}.items():
            mlflow.log_param(name, value)
        X, y, metadata = load_data(args.data_path)
        for name, value in metadata.items():
            mlflow.log_param(name, value)
        X_train, X_test, y_train, y_test = split_data(X, y, args.seed)
        model, messages = fit_model(build_pipeline(X_train, args), X_train, y_train)
        metrics, pred, prob = evaluate_model(model, X_test, y_test)
        for name, value in metrics.items():
            mlflow.log_metric(name, value)
        mlflow.set_tag('converged', str(not messages).lower())
        folder = Path('outputs') / run.info.run_id
        folder.mkdir(parents=True, exist_ok=True)
        save_artifacts(folder, metrics, model, X_train, X_test, y_test, pred, prob, messages)
        for path in folder.iterdir():
            mlflow.log_artifact(str(path), artifact_path='evaluation')
        mlflow.log_artifact(__file__, artifact_path='code')
        mlflow.log_artifact('requirements.txt', artifact_path='code')
        mlflow.sklearn.log_model(model, artifact_path='model',
                                signature=infer_signature(X_train, model.predict(X_train)),
                                input_example=X_train.head(3),
                                pip_requirements=Path('requirements.txt').read_text().splitlines())
        print(json.dumps({'run_id': run.info.run_id, 'metrics': metrics, 'converged': not messages}, indent=2))


if __name__ == '__main__':
    main()


Writing train.py


### Archivo: run_experiments.py

In [4]:
%%writefile run_experiments.py
"""Ejecutar cinco configuraciones, repetir el baseline y comparar evidencia real."""
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient
from train import EXPERIMENT, load_data, sha256, split_data


def main():
    # Un identificador nuevo evita mezclar corridas viejas al volver a ejecutar.
    batch = uuid.uuid4().hex[:12]
    configurations = [
        ('01_baseline', []),
        ('02_C_0_1', ['--C', '0.1']),
        ('03_C_10', ['--C', '10']),
        ('04_sin_balanceo', ['--class-weight', 'none']),
        ('05_tol_0_01', ['--tol', '0.01'])]
    env = os.environ.copy()
    env['PYTHONHASHSEED'] = '42'
    for name, arguments in configurations + [('06_replica_baseline', [])]:
        role = 'replica' if name.startswith('06') else 'comparacion'
        subprocess.run([sys.executable, 'train.py', '--run-name', name,
                        '--batch-id', batch, '--role', role, *arguments], check=True, env=env)
    mlflow.set_tracking_uri('sqlite:///mlflow.db')
    experiment = mlflow.get_experiment_by_name(EXPERIMENT)
    runs = mlflow.search_runs([experiment.experiment_id], filter_string=f"tags.batch_id = '{batch}'")
    assert len(runs) == 6 and (runs.status == 'FINISHED').all(), 'Faltan corridas terminadas.'
    comparison = runs[runs['tags.role'] == 'comparacion'].copy()
    comparison['name'] = comparison['tags.mlflow.runName']
    comparison = comparison.sort_values(['metrics.f1', 'metrics.roc_auc', 'name'], ascending=[False, False, True])
    columns = ['name', 'run_id', 'params.C', 'params.max_iter', 'params.class_weight',
               'params.tol', 'params.seed', 'tags.converged',
               'metrics.accuracy', 'metrics.precision', 'metrics.recall', 'metrics.f1', 'metrics.roc_auc']
    table = comparison[columns]
    assert len(table) == 5
    assert table[['params.C', 'params.max_iter', 'params.class_weight', 'params.tol']].drop_duplicates().shape[0] == 5
    table.to_csv('comparacion.csv', index=False)
    print(table.to_string(index=False))
    # Mismos datos, código, semilla, argumentos y partición en las seis corridas.
    for column in ['params.data_sha256', 'params.train_sha256', 'params.seed']:
        assert runs[column].nunique() == 1, f'Inconsistencia en {column}'
    baseline = runs[runs['tags.mlflow.runName'] == '01_baseline'].iloc[0]
    replica = runs[runs['tags.mlflow.runName'] == '06_replica_baseline'].iloc[0]
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    np.testing.assert_allclose([baseline[f'metrics.{m}'] for m in metrics],
                               [replica[f'metrics.{m}'] for m in metrics], rtol=0, atol=1e-12)
    first = Path('outputs') / baseline.run_id
    second = Path('outputs') / replica.run_id
    assert sha256(first / 'predictions.csv') == sha256(second / 'predictions.csv')
    assert sha256(first / 'split.json') == sha256(second / 'split.json')
    client = MlflowClient()
    for run_id in runs.run_id:
        assert client.list_artifacts(run_id, 'model'), 'Falta el modelo.'
        assert client.list_artifacts(run_id, 'evaluation'), 'Faltan artefactos.'
    best = comparison.iloc[0]
    X, y, _ = load_data('data/online_shoppers_intention.csv')
    _, X_test, _, _ = split_data(X, y, 42)
    restored = mlflow.sklearn.load_model(f'runs:/{best.run_id}/model')
    recorded = pd.read_csv(Path('outputs') / best.run_id / 'predictions.csv')
    np.testing.assert_array_equal(restored.predict(X_test), recorded.prediction)
    np.testing.assert_allclose(restored.predict_proba(X_test)[:, 1], recorded.probability, atol=1e-12, rtol=0)
    evidence = {'batch_id': batch, 'best_run_id': best.run_id,
                'selection_rule': 'f1 descendente, roc_auc descendente, nombre ascendente',
                'same_predictions_and_split': True, 'metric_tolerance': 1e-12,
                'saved_model_verified': True, 'runs': table.to_dict(orient='records')}
    Path('verificacion.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')
    # Comparaciones contra el baseline, un parámetro a la vez.
    base_f1 = float(baseline['metrics.f1'])
    changes = {row['name']: float(row['metrics.f1']) - base_f1
               for _, row in comparison.iterrows() if row['name'] != '01_baseline'}
    influence = max(changes, key=lambda name: abs(changes[name]))
    labels = {'02_C_0_1': 'C=0.1', '03_C_10': 'C=10',
              '04_sin_balanceo': 'class_weight=none', '05_tol_0_01': 'tol=0.01'}
    if max(abs(v) for v in changes.values()) < 1e-12:
        influence_text = 'Ninguna variación produjo un cambio apreciable de F1.'
    else:
        influence_text = (f'El mayor cambio absoluto de F1 respecto del baseline ocurrió con {labels[influence]} '
                          f'(delta={changes[influence]:+.5f}). Esto describe los valores probados; '
                          'no demuestra influencia universal ni significancia estadística. '
                          'Si alguna alternativa no convergió, los deltas también pueden reflejar optimización incompleta.')
    analysis = f'''# Análisis de resultados

Se compararon cinco configuraciones de regresión logística sobre los mismos datos limpios y la misma partición estratificada 80/20 (semilla 42). Antes de ejecutar se eligió F1 de la clase Compra como criterio principal, porque considera precisión y recall; accuracy por sí sola puede resultar engañosa con clases desbalanceadas. ROC AUC y el nombre de corrida resuelven empates, en ese orden.

La mejor configuración observada fue {best['name']} (run_id: {best.run_id}), con C={best['params.C']}, max_iter={best['params.max_iter']}, class_weight={best['params.class_weight']} y tol={best['params.tol']}. Obtuvo F1={best['metrics.f1']:.5f}, ROC AUC={best['metrics.roc_auc']:.5f}, precisión={best['metrics.precision']:.5f}, recall={best['metrics.recall']:.5f} y accuracy={best['metrics.accuracy']:.5f}. Es la mejor entre estas cinco alternativas según la regla fijada, no necesariamente la mejor posible.

{influence_text} Cada alternativa cambió un solo parámetro frente al baseline; max_iter se mantuvo en 1000 y no se evaluó su influencia. Deltas de F1: {', '.join(labels[k] + '=' + format(v, '+.5f') for k, v in changes.items())}.

La repetición del baseline reprodujo métricas dentro de 1e-12, predicciones y partición; también se verificaron las predicciones del modelo guardado. Limitaciones: solo se usó una partición y no hay intervalos de confianza. Como se comparó sobre el holdout original para conservar el baseline, la selección reutiliza ese conjunto y puede ser optimista; una evaluación final requeriría validación cruzada y un test adicional intacto. La regresión logística tiene límites para relaciones no lineales. PageValues se conservó por fidelidad, pero debe revisarse su disponibilidad al instante de una predicción real. Corridas sin convergencia: {int((comparison['tags.converged'] != 'true').sum())}; revisar convergence.json antes de confiar en ellas.

Evidencia: comparacion.csv y verificacion.json, grupo {batch}. Adjuntar las capturas reales de la tabla y de Compare en MLflow, con parámetros y métricas legibles.
'''
    assert len(analysis.split()) <= 450
    Path('analisis_resultados.md').write_text(analysis, encoding='utf-8')
    print('\nVerificación correcta. Grupo:', batch)
    print(analysis)


if __name__ == '__main__':
    main()


Writing run_experiments.py


### Archivo: README.md

In [5]:
%%writefile README.md
# TC5061: entrenamiento parametrizado con MLflow

## Qué conserva del baseline

Dataset Online Shoppers Purchasing Intention de UCI, objetivo Revenue y 17 predictores. Se eliminan duplicados completos, Weekend y Revenue se convierten a 0/1, y se conserva la partición estratificada 80/20 con semilla 42. Las columnas numéricas incluyen los códigos OperatingSystems, Browser, Region y TrafficType, igual que en el baseline. No se eliminan outliers ni PageValues. Se utiliza imputación por mediana y RobustScaler para números, moda y OneHotEncoder para categorías, y LogisticRegression con solver lbfgs y penalización L2. El pipeline se ajusta solo con entrenamiento.

## Google Colab, paso a paso

1. Abre https://colab.research.google.com/ y elige Archivo > Subir notebook. Selecciona `TC5061_MLflow_Colab.ipynb`.
2. Guarda una copia en Drive. Elige un entorno Python 3 con CPU; no se necesita GPU. El paso 4 usa uv 0.6.17 para preparar Python 3.12.10 en `.venv312`, incluso si el kernel de Colab utiliza Python 3.13. Así evita depender de venv/ensurepip del sistema y mantiene las versiones del entrenamiento. No necesitas reiniciar la sesión.
3. Ejecuta las celdas en orden. La carpeta de trabajo será `/content/tc5061_mlflow`. Las primeras celdas generan requirements.txt, train.py, run_experiments.py y este README. La instalación necesita conexión a internet.
4. Revisa la salida de `train.py --help`. Ejecuta la exploración y después la celda de cinco configuraciones. UCI se descarga una sola vez. Si UCI no responde, sube el CSV original con el nombre `online_shoppers_intention.csv` a la carpeta `data/` usando el panel Archivos; vuelve a ejecutar. No uses otro dataset.
5. Espera la confirmación de verificación. Se generan seis runs: cinco distintos y una réplica, todos en `TC5061_Online_Shoppers_Regresion_Logistica_Comparacion`. Al repetir el lote se genera otra etiqueta batch_id, para no mezclar evidencia anterior.
6. Ejecuta la celda que inicia MLflow y espera el mensaje del servidor. La siguiente celda abre un enlace mediante el proxy de Colab. Mantén el entorno conectado; `localhost:5000` en tu computadora no apunta a Colab.
7. En MLflow abre Experiments, selecciona el experimento y filtra con `tags.batch_id = 'VALOR'`, usando el valor impreso. Agrega `tags.role = 'comparacion'` si aparece la réplica. Deben quedar cinco runs.
8. En Columns/Columns to display selecciona nombre, C, max_iter, class_weight, tol, seed, accuracy, precision, recall, f1 y roc_auc. Ordena f1 de mayor a menor. Ajusta el ancho del navegador o reduce el zoom para ver encabezados y las cinco filas. Captura la tabla completa.
9. Marca las casillas de esas cinco corridas y pulsa Compare. Revisa la tabla de Parameters y Metrics; captura ambas secciones con valores y nombres legibles. Si necesitas dos imágenes para que el texto sea legible, toma dos. No reemplaces esta evidencia por la tabla de pandas.
10. Abre una corrida > Artifacts. Comprueba `evaluation/confusion_matrix.png`, metrics.json, predictions.csv, split.json, convergence.json y `model/MLmodel`. Puedes capturar esta vista como evidencia adicional.
11. Guarda las capturas como `tabla_mlflow.png`, `comparacion_parametros.png` y `comparacion_metricas.png`. Sube esas imágenes al notebook en la celda indicada. Esta celda las guarda e inserta en el notebook; vuelve a guardar la copia en Drive.
12. Revisa el análisis generado a partir de las métricas reales. Tiene menos de 450 palabras; verifica que quepa en una cuartilla con el formato solicitado por tu profesor. Descarga el paquete de evidencia y el notebook ejecutado (Archivo > Descargar > .ipynb). Entrega ambos: el notebook muestra las salidas y las imágenes; el ZIP preserva datos, código, modelos y registros.

Si el enlace del proxy no abre, permite ventanas emergentes para Colab y ejecuta la celda del iframe. Si sigue fallando, descarga la evidencia y abre MLflow localmente con las instrucciones de abajo. Si la tabla está vacía, verifica la URI, el experimento y el batch_id. No borres la base para corregir el filtro. El registro queda en la carpeta temporal de Colab: descárgalo antes de desconectar.

## Ejecución local desde una terminal

Usa Python 3.11 o 3.12. Abre una terminal en la carpeta que contiene train.py y requirements.txt.

```text
python -m venv .venv
```

Activa el entorno: Windows PowerShell ` .\.venv\Scripts\Activate.ps1`; macOS/Linux `source .venv/bin/activate`. Si PowerShell impide la activación, usa directamente `.venv\Scripts\python.exe` en lugar de `python`, sin cambiar políticas del sistema.

```text
python -m pip install -r requirements.txt
python train.py --help
python train.py
python train.py --C 0.1 --max-iter 1500 --class-weight none --tol 0.0001 --seed 42 --run-name ejemplo
python run_experiments.py
python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Abre http://127.0.0.1:5000. Se recomienda generar las corridas en esta carpeta local para que las rutas de artefactos correspondan a tu equipo. Para consultar el ZIP de Colab en Linux, extráelo en `/content/tc5061_mlflow`, la misma ruta original, y ejecuta la interfaz desde ahí; MLflow guarda rutas absolutas a los artefactos. En Windows, puedes abrir la tabla de la base extraída, pero para navegar los modelos en la UI vuelve a ejecutar el script localmente; los archivos de modelo sí permanecen dentro del ZIP.

## Parámetros y comparación

| Argumento | Valor por defecto | Significado |
|---|---|---|
| --C | 1.0 | Inverso de regularización; menor C regulariza más |
| --max-iter | 1000 | Tope de iteraciones; no garantiza convergencia |
| --class-weight | balanced | Compensa el desbalance; none desactiva ponderación |
| --tol | 0.0001 | Tolerancia de parada del optimizador |
| --seed | 42 | Semilla de Python, NumPy, división y modelo |

Se prueban baseline, C=0.1, C=10, class_weight=none y tol=0.01; el resto permanece fijo. Se elige por F1 de Compra, después ROC AUC y nombre para empates. Los cambios frente al baseline permiten hablar de influencia dentro del rango ensayado. La tolerancia afecta la optimización y no necesariamente mejorará F1. max_iter se expone pero no se modifica en estas cinco configuraciones.

## Reproducibilidad y alcance

La réplica compara las cinco métricas (tolerancia absoluta 1e-12), el hash de predictions.csv y split.json. Los hashes de datos y código se guardan por corrida. Se limitan los hilos numéricos y se fija PYTHONHASHSEED al lanzar el lote. El modelo guardado se carga y se comparan predicciones y probabilidades con las registradas. Se preservan requirements.txt, `pip freeze` y versión de Python; las dependencias directas están fijadas y environment_freeze.txt documenta también las transitivas del entorno real. Reproducir exige el mismo CSV, código y entorno; no se promete igualdad entre arquitecturas o versiones distintas.

El holdout original se reutiliza para comparar por fidelidad al baseline: la estimación tras seleccionar puede ser optimista. Para uso real se necesita validación y un test independiente. El análisis solo usa corridas terminadas del lote; no incluye números inventados. Las capturas se obtienen de la interfaz real después de ejecutar. Revisar `tags.converged` y convergence.json; una ejecución terminada no implica convergencia.

## Relación con la rúbrica

| Criterio | Evidencia a entregar |
|---|---|
| Migración, 20% | train.py con funciones de carga, división, pipeline, ajuste, evaluación, artefactos y main; baseline original reservado para exploración |
| Parametrización, 15% | Cuatro hiperparámetros más seed, defaults y --help |
| Tracking, 25% | start_run, log_param, log_metric, log_artifact y sklearn.log_model; experimento descriptivo |
| Corridas y comparación, 20% | Cinco configuraciones distintas, comparacion.csv y capturas de tabla y Compare en MLflow |
| Reproducibilidad, 10% | Réplica verificada, hashes, requirements con versiones y README |
| Análisis, 10% | analisis_resultados.md con ganador observado, deltas y limitaciones, acompañado por capturas |

## Referencias

- Baseline de Martha Alicia Villalobos Facundo: `TC5061_Notebook_Baseline_VillalobosMartha.ipynb`.
- UCI: https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset
- MLflow Tracking: https://mlflow.org/docs/latest/ml/tracking/
- Registro de modelos sklearn: https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html
- argparse: https://docs.python.org/3/library/argparse.html
- Proxy oficial de Colab: https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py


Writing README.md


## 4. Instalar las librerías y verificar la sintaxis

Esta celda instala uv 0.6.17 y prepara un intérprete Python 3.12.10 separado, aunque el kernel de Colab use Python 3.13. uv crea el entorno sin depender de venv/ensurepip del sistema. Las librerías del entrenamiento se instalan en `.venv312`.

In [6]:

from pathlib import Path
import os
import subprocess
import sys

WORK = Path('/content/tc5061_mlflow')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)


def ejecutar_instalacion(comando):
    resultado = subprocess.run(comando, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    print(resultado.stdout)
    resultado.check_returncode()


# uv crea el entorno sin depender del módulo venv/ensurepip de Colab.
# Python 3.12.10 mantiene compatibles las versiones de requirements.txt.
ejecutar_instalacion([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.6.17'])
os.environ['UV_PYTHON_INSTALL_DIR'] = '/content/tc5061_python'
PYTHON = str(WORK / '.venv312/bin/python')
if not Path(PYTHON).exists():
    ejecutar_instalacion([sys.executable, '-m', 'uv', 'venv', '--python',
                         '3.12.10', '--seed', str(WORK / '.venv312')])
ejecutar_instalacion([PYTHON, '--version'])
ejecutar_instalacion([PYTHON, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
ejecutar_instalacion([PYTHON, '-m', 'pip', 'check'])
ejecutar_instalacion([PYTHON, '-m', 'py_compile', 'train.py', 'run_experiments.py'])
print(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
print('Paso 4 completado. Continúa con el paso 5; no necesitas reiniciar Colab.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 48.4 MB/s eta 0:00:00

Using CPython 3.12.10
Creating virtual environment with seed packages at: .venv312
 + pip==26.2.1
Activate with: source .venv312/bin/activate

Python 3.12.10


No broken requirements found.


alembic==1.20.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
blinker==1.9.0
cachetools==5.5.2
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
contourpy==1.4.0
cryptography==50.0.2
cycler==0.12.1
databricks-sdk==0.151.0
docker==7.2.0
fastapi==0.143.0
Flask==3.1.3
fonttools==4.66.1
gitdb==4.0.12
GitPython==3.2.0
google-auth==2.61.0
graphene==3.4.3
graphql-core==3.2.13
graphql-relay==3.2.0
greenlet==3.5.6
gunicorn==23.0.0
h11==0.16.0
idna==3.20
importlib_metadata==8.9.0
itsdangerous==2.2.0
Jinja2==3.1.6
joblib==1.6.0
kiwisolver==1.5.1
Mako==1.4.3
Markdown==3.11
MarkupSafe==3.0.4
matplotlib==3.10.1
mlflow==2.22.0
mlflow-skinny==2.22.0
numpy==2.1.3
opentelemetry-api==1.4

## 5. Revisar la ayuda y los argumentos

`argparse` permite ejecutar sin argumentos o modificar el modelo desde una terminal. Hay cuatro hiperparámetros: C, max_iter, class_weight y tol, más la semilla. Los defaults reproducen la configuración del baseline. La ayuda muestra valores por defecto y significado; los valores no válidos detienen la ejecución con un mensaje.

| Parámetro | Valor por defecto | Interpretación |
|---|---|---|
| C | 1.0 | Inverso de la regularización; un valor pequeño regulariza más |
| max_iter | 1000 | Tope de iteraciones del optimizador |
| class_weight | balanced | Pondera clases según frecuencia; none lo desactiva |
| tol | 0.0001 | Tolerancia de convergencia |
| seed | 42 | Semilla para Python, NumPy, partición y modelo |

Ejecutar `python train.py` registra una corrida con defaults. Desde Colab usé el intérprete de `.venv312`, mediante la variable PYTHON preparada en el paso.

In [9]:
import os
os.environ["MPLBACKEND"] = "Agg"

resultado = subprocess.run(
    [PYTHON, str(WORK / "train.py"), "--help"],
    cwd=str(WORK),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(resultado.stdout)
resultado.check_returncode()


usage: train.py [-h] [--C C] [--max-iter MAX_ITER]
                [--class-weight {balanced,none}] [--tol TOL] [--seed SEED]
                [--data-path DATA_PATH] [--tracking-uri TRACKING_URI]
                [--experiment EXPERIMENT] [--run-name RUN_NAME]
                [--batch-id BATCH_ID] [--role {comparacion,replica}]

Entrenamiento reproducible del baseline Online Shoppers. Ejecutar: python
train.py.

options:
  -h, --help            show this help message and exit
  --C C                 Inverso de la regularización; debe ser positivo
                        (default: 1.0)
  --max-iter MAX_ITER   Máximo de iteraciones del optimizador (default: 1000)
  --class-weight {balanced,none}
                        Ponderación de clases (default: balanced)
  --tol TOL             Tolerancia de convergencia positiva (default: 0.0001)
  --seed SEED           Semilla para Python, NumPy y scikit-learn (default:
                        42)
  --data-path DATA_PATH
                        CS

## 6. Explorar los datos sin entrenar

Esta celda reutiliza `load_data` del script para descargar y limpiar, sin duplicar la lógica. Muestra el conteo de duplicados, las dimensiones y el desbalance. El CSV original se descarga desde UCI y se guarda para que todas las corridas usen exactamente los mismos bytes. Si UCI no responde, utiliza el panel Archivos para subir el CSV original a `data/online_shoppers_intention.csv`


In [10]:
exploration = """
from train import load_data
X, y, metadata = load_data('data/online_shoppers_intention.csv')
print('Control de datos:', metadata)
print('Predictores:', X.shape)
print(X.head().to_string())
print('Conteo de Revenue:', y.value_counts().sort_index().to_dict())
print('Porcentaje de Compra:', round(y.mean() * 100, 2))
"""
subprocess.run([PYTHON, '-c', exploration], check=True)


CompletedProcess(args=['/content/tc5061_mlflow/.venv312/bin/python', '-c', "\nfrom train import load_data\nX, y, metadata = load_data('data/online_shoppers_intention.csv')\nprint('Control de datos:', metadata)\nprint('Predictores:', X.shape)\nprint(X.head().to_string())\nprint('Conteo de Revenue:', y.value_counts().sort_index().to_dict())\nprint('Porcentaje de Compra:', round(y.mean() * 100, 2))\n"], returncode=0)

## 7. Ejecutar cinco configuraciones distintas y una réplica

**Regla definida antes de observar resultados:** seleccionar el F1 de Compra más alto; desempatar por ROC AUC más alto y después nombre de corrida. F1 considera precisión y recall; accuracy puede ocultar el desbalance.

| Corrida | C | max_iter | class_weight | tol | seed |
|---|---:|---:|---|---:|---:|
| 01_baseline | 1 | 1000 | balanced | 0.0001 | 42 |
| 02_C_0_1 | 0.1 | 1000 | balanced | 0.0001 | 42 |
| 03_C_10 | 10 | 1000 | balanced | 0.0001 | 42 |
| 04_sin_balanceo | 1 | 1000 | none | 0.0001 | 42 |
| 05_tol_0_01 | 1 | 1000 | balanced | 0.01 | 42 |
| 06_replica_baseline | 1 | 1000 | balanced | 0.0001 | 42 |

Las primeras cinco son la comparación; la sexta comprueba reproducibilidad. Cada alternativa modifica un solo parámetro respecto al baseline. max_iter permanece fijo y no se podrá concluir nada sobre su influencia. No se modifica la semilla ni la partición para evitar confundir esos efectos con los hiperparámetros.

Todo se registra bajo `TC5061_Online_Shoppers_Regresion_Logistica_Comparacion`, en SQLite. Cada ejecución del lote crea un batch_id distinto.


In [11]:
subprocess.run([PYTHON, 'run_experiments.py'], check=True)


CompletedProcess(args=['/content/tc5061_mlflow/.venv312/bin/python', 'run_experiments.py'], returncode=0)

## 8. Comprobar comparación, reproducibilidad y modelo guardado

Cada run guarda parámetros con `log_param`, cinco métricas con `log_metric`, artefactos con `log_artifact` y el pipeline con `mlflow.sklearn.log_model`, dentro de `start_run`. `convergence.json` indica advertencias; revisa `tags.converged`.


In [12]:
import json
import pandas as pd
from IPython.display import display, Markdown

verification = json.loads(Path('verificacion.json').read_text())
BATCH_ID = verification['batch_id']
table = pd.read_csv('comparacion.csv')
display(table)
print('Grupo para filtrar en MLflow:', BATCH_ID)
print('Mismas predicciones y partición:', verification['same_predictions_and_split'])
print('Modelo guardado verificado:', verification['saved_model_verified'])


,name,run_id,params.C,params.max_iter,params.class_weight,params.tol,params.seed,tags.converged,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,05_tol_0_01,4be0d9bd6ce347ff805f2e69444213b5,1.0,1000,balanced,0.0100,42,True,0.852929,0.519726,0.793194,0.627979,0.912444
1,02_C_0_1,618f2806c704462ab154857422bc6f11,0.1,1000,balanced,0.0001,42,False,0.852519,0.519031,0.785340,0.625000,0.911639
2,01_baseline,a2ddb8f410ef4f6da1c6742df2af899e,1.0,1000,balanced,0.0001,42,True,0.851290,0.516351,0.785340,0.623053,0.911033
3,03_C_10,22f46228616742d1a6250fbf9b9af3f8,10.0,1000,balanced,0.0001,42,True,0.851290,0.516351,0.785340,0.623053,0.911018
4,04_sin_balanceo,c5d68e13890a4d9f90a562ceac0e7d84,1.0,1000,none,0.0001,42,False,0.889390,0.771845,0.416230,0.540816,0.901724


Grupo para filtrar en MLflow: a06886db54ba
Mismas predicciones y partición: True
Modelo guardado verificado: True


## 9. Iniciar la interfaz de MLflow en Colab



In [13]:
import time
import urllib.request

if 'mlflow_process' not in globals() or mlflow_process.poll() is not None:
    ui_log = open('mlflow_ui.log', 'a', encoding='utf-8')
    mlflow_process = subprocess.Popen(
        [PYTHON, '-m', 'mlflow', 'ui', '--backend-store-uri', 'sqlite:///mlflow.db',
         '--host', '0.0.0.0', '--port', '5000'],
        stdout=ui_log, stderr=subprocess.STDOUT, cwd=WORK)
ready = False
for attempt in range(45):
    try:
        with urllib.request.urlopen('http://127.0.0.1:5000', timeout=2) as response:
            ready = response.status == 200
        if ready:
            break
    except Exception:
        time.sleep(1)
if not ready:
    print(Path('mlflow_ui.log').read_text()[-6000:])
    raise RuntimeError('MLflow no inició. Revisa el log antes de continuar.')
print('Servidor listo. Usa el enlace de la siguiente celda.')


Servidor listo. Usa el enlace de la siguiente celda.


### MLflow con el proxy de Colab


In [14]:
from google.colab import output
output.serve_kernel_port_as_window(5000)
print("Filtro de tabla: tags.batch_id = '" + BATCH_ID + "' and tags.role = 'comparacion'")


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Filtro de tabla: tags.batch_id = 'a06886db54ba' and tags.role = 'comparacion'


### Alternativa


In [15]:
output.serve_kernel_port_as_iframe(5000, height=800)


<IPython.core.display.Javascript object>